# Day 21 — Player API Pagination and Filtering

Goal: inspect the current Django Player API, add reusable pagination validation and unit tests, and generate documentation from actual results.

Run from the repository root using the `ipl-player-performance` kernel. This notebook does not overwrite `views.py`; integrate the helper only after confirming your existing SQL and real player-name column.

In [1]:
from pathlib import Path
import os, sys, subprocess, csv, shutil
from datetime import datetime

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    roots = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if roots:
        REPO_ROOT = roots[0]
DJANGO_ROOT = REPO_ROOT / "django_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
VIEWS_PATH = DJANGO_ROOT / "ipl_api" / "views.py"
URLS_PATH = DJANGO_ROOT / "ipl_api" / "urls.py"
DOCS_DIR = REPO_ROOT / "docs"
REPORT_DIR = REPO_ROOT / "reports" / "day21"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_ROOT)
for p in [MANAGE_PY, VIEWS_PATH, URLS_PATH, DJANGO_ROOT / "config" / "settings.py"]:
    print(("FOUND " if p.exists() else "MISSING "), p)
if not all(p.exists() for p in [MANAGE_PY, VIEWS_PATH, URLS_PATH]):
    raise FileNotFoundError("Required Django files missing. Complete Days 18–20 first.")


Repository: D:\Git\Personal\ipl-player-performance
FOUND  D:\Git\Personal\ipl-player-performance\django_api\manage.py
FOUND  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\views.py
FOUND  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
FOUND  D:\Git\Personal\ipl-player-performance\django_api\config\settings.py


In [2]:
print("=== ipl_api/urls.py ===")
print(URLS_PATH.read_text(encoding="utf-8"))
print("=== ipl_api/views.py ===")
print(VIEWS_PATH.read_text(encoding="utf-8"))


=== ipl_api/urls.py ===
from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView

urlpatterns = [
    path("players/", PlayerListAPIView.as_view(), name="player-list"),
    path("players/<str:player_name>/", PlayerDetailAPIView.as_view(), name="player-detail"),
]

=== ipl_api/views.py ===
from django.db import connection
from django.http import Http404
from rest_framework import status
from rest_framework.response import Response
from rest_framework.views import APIView

# Table and column names detected from the MySQL staging schema.
BATTING_TABLE = 'stg_ipl_batting_player_match'
BOWLING_TABLE = 'stg_ipl_bowling_player_match'
BAT_NAME = 'player_name'
BOWL_NAME = 'player_name'
BAT_RUNS = 'runs'
BAT_MATCH = 'match_id'
BAT_BALLS = 'balls_faced'
BAT_FOURS = 'fours'
BAT_SIXES = 'sixes'
BOWL_WICKETS = 'wickets'
BOWL_RUNS = 'runs_conceded'
BOWL_OVERS = 'overs'
BOWL_MATCH = 'match_id'

def q(identifier):
    if not identifier or not identifier.replace("_", "").is

## Recommended API contract

- `GET /api/players/?page=1&page_size=20`
- `GET /api/players/?search=virat`
- `page` must be an integer >= 1.
- `page_size` must be between 1 and 100.
- Search must use parameterized SQL, never string concatenation.
- Suggested response: `count`, `page`, `page_size`, `total_pages`, `results`.

For large datasets, prefer SQL `LIMIT`/`OFFSET`. This notebook creates a reusable helper and tests, but does not rewrite the existing Player List view until its query and actual column names are confirmed.

In [3]:
HELPER_PATH = DJANGO_ROOT / "ipl_api" / "pagination_utils.py"
if HELPER_PATH.exists():
    backup = HELPER_PATH.with_name("pagination_utils.py.backup_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    shutil.copy2(HELPER_PATH, backup)
    print("Backup created:", backup)

helper_lines = [
    "from math import ceil",
    "from django.core.exceptions import ValidationError",
    "",
    "def parse_pagination_params(query_params, default_page_size=20, max_page_size=100):",
    "    try:",
    "        page = int(query_params.get('page', 1))",
    "        page_size = int(query_params.get('page_size', default_page_size))",
    "    except (TypeError, ValueError):",
    "        raise ValidationError('page and page_size must be integers.')",
    "    if page < 1:",
    "        raise ValidationError('page must be greater than or equal to 1.')",
    "    if page_size < 1 or page_size > max_page_size:",
    "        raise ValidationError(f'page_size must be between 1 and {max_page_size}.')",
    "    return page, page_size",
    "",
    "def paginate_records(records, page, page_size):",
    "    total = len(records)",
    "    start = (page - 1) * page_size",
    "    return {",
    "        'count': total,",
    "        'page': page,",
    "        'page_size': page_size,",
    "        'total_pages': ceil(total / page_size) if total else 0,",
    "        'results': records[start:start + page_size],",
    "    }",
]
helper_code = "\n".join(helper_lines) + "\n"
HELPER_PATH.write_text(helper_code, encoding="utf-8")
compile(helper_code, str(HELPER_PATH), "exec")
print("Created:", HELPER_PATH)

TESTS_DIR = DJANGO_ROOT / "ipl_api" / "tests"
TESTS_DIR.mkdir(parents=True, exist_ok=True)
(TESTS_DIR / "__init__.py").touch(exist_ok=True)
TEST_PATH = TESTS_DIR / "test_pagination_utils_day21.py"
if TEST_PATH.exists():
    backup_test = TEST_PATH.with_name("test_pagination_utils_day21.py.backup_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    shutil.copy2(TEST_PATH, backup_test)
    print("Test backup:", backup_test)

test_lines = [
    "from django.test import SimpleTestCase",
    "from django.core.exceptions import ValidationError",
    "from ipl_api.pagination_utils import parse_pagination_params, paginate_records",
    "",
    "class PaginationHelperTests(SimpleTestCase):",
    "    def test_defaults(self):",
    "        self.assertEqual(parse_pagination_params({}), (1, 20))",
    "    def test_valid_values(self):",
    "        self.assertEqual(parse_pagination_params({'page': '2', 'page_size': '10'}), (2, 10))",
    "    def test_rejects_non_integer(self):",
    "        with self.assertRaises(ValidationError):",
    "            parse_pagination_params({'page': 'abc'})",
    "    def test_rejects_zero_page(self):",
    "        with self.assertRaises(ValidationError):",
    "            parse_pagination_params({'page': '0'})",
    "    def test_rejects_oversized_page_size(self):",
    "        with self.assertRaises(ValidationError):",
    "            parse_pagination_params({'page_size': '101'})",
    "    def test_paginates_records(self):",
    "        records = [{'id': i} for i in range(1, 6)]",
    "        payload = paginate_records(records, page=2, page_size=2)",
    "        self.assertEqual(payload['count'], 5)",
    "        self.assertEqual(payload['total_pages'], 3)",
    "        self.assertEqual(payload['results'], [{'id': 3}, {'id': 4}])",
]
test_code = "\n".join(test_lines) + "\n"
TEST_PATH.write_text(test_code, encoding="utf-8")
print("Created:", TEST_PATH)


Created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\pagination_utils.py
Created: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\tests\test_pagination_utils_day21.py


In [4]:
env = os.environ.copy()
check = subprocess.run([sys.executable, str(MANAGE_PY), "check"], cwd=str(DJANGO_ROOT), capture_output=True, text=True, env=env)
print("=== python manage.py check ===")
print("Return code:", check.returncode)
print(check.stdout)
if check.stderr:
    print(check.stderr)

tests = subprocess.run(
    [sys.executable, str(MANAGE_PY), "test", "ipl_api.tests.test_pagination_utils_day21", "--verbosity", "2"],
    cwd=str(DJANGO_ROOT), capture_output=True, text=True, env=env
)
print("=== Pagination helper tests ===")
print("Return code:", tests.returncode)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)


=== python manage.py check ===
Return code: 0
System check identified no issues (0 silenced).

=== Pagination helper tests ===
Return code: 0
Found 6 test(s).
Skipping setup of unused database(s): default.
System check identified no issues (0 silenced).

test_defaults (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_defaults) ... ok
test_paginates_records (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_paginates_records) ... ok
test_rejects_non_integer (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_rejects_non_integer) ... ok
test_rejects_oversized_page_size (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_rejects_oversized_page_size) ... ok
test_rejects_zero_page (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_rejects_zero_page) ... ok
test_valid_values (ipl_api.tests.test_pagination_utils_day21.PaginationHelperTests.test_valid_values) ... ok

----------------------------------

In [5]:
def tail(text, count=30):
    return "\\n".join((text or "").splitlines()[-count:])

with (REPORT_DIR / "task_status.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["item", "value"])
    writer.writerows([
        ("generated_at", datetime.now().isoformat(timespec="seconds")),
        ("repository_root", str(REPO_ROOT)),
        ("helper_path", str(HELPER_PATH)),
        ("test_path", str(TEST_PATH)),
        ("django_check_return_code", str(check.returncode)),
        ("pagination_tests_return_code", str(tests.returncode)),
    ])

doc_lines = [
    "# Day 21 — Player API Pagination and Filtering",
    "",
    "- Generated: " + datetime.now().isoformat(timespec="seconds"),
    "- Repository: `" + str(REPO_ROOT) + "`",
    "- Helper: `" + str(HELPER_PATH) + "`",
    "- Tests: `" + str(TEST_PATH) + "`",
    "- Django check return code: **" + str(check.returncode) + "**",
    "- Pagination test return code: **" + str(tests.returncode) + "**",
    "",
    "## Django check output",
    "",
    "```text",
    tail(check.stdout + chr(10) + check.stderr, 20),
    "```",
    "",
    "## Pagination test output",
    "",
    "```text",
    tail(tests.stdout + chr(10) + tests.stderr, 35),
    "```",
    "",
    "## Integration status",
    "",
    "This notebook creates and tests a reusable helper. It does not rewrite the existing Player List view. Integrate the helper into the existing API using the actual SQL query and player-name column. For large datasets, use parameterized SQL LIMIT/OFFSET.",
    "",
    "## API contract",
    "",
    "- `GET /api/players/?page=1&page_size=20`",
    "- `GET /api/players/?search=virat`",
    "- `page >= 1`; `1 <= page_size <= 100`",
    "- Suggested response keys: `count`, `page`, `page_size`, `total_pages`, `results`",
]
(DOCS_DIR / "day21_player_api_pagination_filtering.md").write_text("\n".join(doc_lines) + "\n", encoding="utf-8")
print("Saved:", REPORT_DIR / "task_status.csv")
print("Saved:", DOCS_DIR / "day21_player_api_pagination_filtering.md")


Saved: D:\Git\Personal\ipl-player-performance\reports\day21\task_status.csv
Saved: D:\Git\Personal\ipl-player-performance\docs\day21_player_api_pagination_filtering.md


## Day 21 completion checklist

- [ ] Django system check passes.
- [ ] All six pagination helper tests pass.
- [ ] Pagination is integrated into the existing Player List API.
- [ ] Search uses the actual player-name column and parameterized SQL.
- [ ] Test valid, invalid, empty and out-of-range parameters.
- [ ] Review generated documentation and reports.

Helper tests alone do not prove the endpoint is integrated. Confirm the actual endpoint response before marking Day 21 complete.

**Next:** Day 22 — EDA API endpoints.